# Module 16 — Exercises

These exercises rebuild a neural network layer by layer: activations and their derivatives, parameter and FLOP
counting, the stable softmax cross-entropy and its gradient, a `Dense` layer's backward pass with a gradient checker,
initialisation and variance propagation, batch normalisation, inverted dropout, a conceptual question on depth and
the He factor, convolution as a Toeplitz matrix, and a mini-project that trains a regularised digit classifier and
reports a bootstrap confidence interval. Only NumPy, SciPy and matplotlib are allowed — no PyTorch, no JAX. Data
matrices are `(n_samples, n_features)` and a layer computes `X @ W + b`. The concept guide (`README.md`) has every formula.

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf
rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (7, 4)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def central_diff(f, x, h=1e-6):
    # elementwise central difference of a vectorised scalar function f at the points x
    return (f(x + h) - f(x - h)) / (2 * h)

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — Activation functions and their derivatives  `★☆☆`

Implement `relu`, `softplus` and `gelu` and the derivatives `d_relu`, `d_sigmoid`, `d_tanh`, `d_softplus`, `d_gelu`
(all vectorised over NumPy arrays; `sigmoid` is given in the setup cell, and `erf` is imported from `scipy.special`):

$$
\mathrm{softplus}(z) = \ln(1 + e^z),\quad \mathrm{softplus}'(z) = \sigma(z); \qquad
\mathrm{GELU}(z) = z\,\Phi(z),\quad \mathrm{GELU}'(z) = \Phi(z) + z\,\varphi(z),
$$

with $\Phi(z) = \tfrac12\big(1 + \mathrm{erf}(z/\sqrt2)\big)$ the standard normal CDF and $\varphi(z) = e^{-z^2/2}/\sqrt{2\pi}$ its density.
`softplus` must not overflow for `z = 1000` (write it as $\max(z, 0) + \ln(1 + e^{-|z|})$). The checks compare every
derivative with central differences; two hidden checks ask for $\mathrm{GELU}(1.5)$ and $\mathrm{softplus}'(-3)$.

In [ ]:
def relu(z):
    # YOUR CODE HERE
    return None

def d_relu(z):
    # YOUR CODE HERE
    return None

def d_sigmoid(z):
    # YOUR CODE HERE (sigmoid is already defined in the setup cell)
    return None

def d_tanh(z):
    # YOUR CODE HERE
    return None

def softplus(z):
    # log(1 + e^z), written so that it does not overflow for z = 1000
    # YOUR CODE HERE
    return None

def d_softplus(z):
    # YOUR CODE HERE
    return None

def gelu(z):
    # z * Phi(z) with Phi the standard normal CDF: Phi(z) = 0.5 * (1 + erf(z / sqrt(2)))
    # YOUR CODE HERE
    return None

def d_gelu(z):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
zs = np.array([-3.0, -0.7, 0.2, 1.5, 4.0])
def _safe(fn, *a):
    try:
        out = fn(*a)
        return None if out is None else np.asarray(out, dtype=float)
    except Exception:
        return None
check_close('relu values', _safe(relu, zs), np.maximum(zs, 0))
check_close("relu' = 1[z > 0]", _safe(d_relu, zs), (zs > 0).astype(float))
check_close("sigmoid' vs central differences", _safe(d_sigmoid, zs), central_diff(sigmoid, zs), tol=1e-6)
check_close("tanh' vs central differences", _safe(d_tanh, zs), central_diff(np.tanh, zs), tol=1e-6)
sp = _safe(softplus, zs)
check_close('softplus values', sp, np.log1p(np.exp(zs)), tol=1e-9)
check('softplus(1000) is finite and equals 1000', (lambda v: v is not None and np.isfinite(v) and abs(float(v) - 1000.0) < 1e-9)(_safe(softplus, np.array(1000.0))))
check_close("softplus' vs central differences", _safe(d_softplus, zs), (central_diff(softplus, zs) if sp is not None else None), tol=1e-6)
ge = _safe(gelu, zs)
check_close('gelu(0) = 0 and gelu(4) ~ 4', _safe(gelu, np.array([0.0, 4.0])), np.array([0.0, 4.0 * 0.5 * (1 + erf(4.0 / np.sqrt(2)))]), tol=1e-9)
check_close("gelu' vs central differences", _safe(d_gelu, zs), (central_diff(gelu, zs) if ge is not None else None), tol=1e-6)
check_hash('hidden: GELU(1.5)', (float(ge[3]) if ge is not None else None), 'f03af073f3a641fa')
check_hash("hidden: softplus'(-3)", (float(_safe(d_softplus, np.array(-3.0))) if _safe(d_softplus, np.array(-3.0)) is not None else None), 'e5d3e87c0a26675a')

## Exercise 2 — Parameter count, FLOPs and shapes  `★☆☆`

For an MLP with layer sizes `sizes = [n_in, h_1, ..., n_out]` implement

* `count_params(sizes)` — the total number of weights and biases, $\sum_l (n_l + 1)\, n_{l+1}$ (return an `int`);
* `forward_flops(sizes, batch_size)` — the FLOPs of all matrix products in one forward pass, counting each
  multiply-add as 2 FLOPs and ignoring biases and activations (an `int`);
* `forward_shapes(sizes, batch_size)` — the list of shapes `(batch_size, n_{l+1})` of $Z_1, \dots, Z_L$ as tuples.

Hidden checks: the parameter count of a 784 → 512 → 256 → 10 network, and the forward FLOPs of 64 → 128 → 10 at batch size 32.

In [ ]:
def count_params(sizes):
    # sizes = [n_in, h_1, ..., h_{L-1}, n_out]; every layer has a weight matrix and a bias vector
    # YOUR CODE HERE
    return None

def forward_flops(sizes, batch_size):
    # multiply-adds of all matrix products for one forward pass of a batch, counted as 2 FLOPs each
    # (ignore the bias additions and the activations)
    # YOUR CODE HERE
    return None

def forward_shapes(sizes, batch_size):
    # list of the shapes (as tuples) of Z_1, Z_2, ..., Z_L for a batch of batch_size rows
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    p_xor = count_params([2, 2, 1]); p_dig = count_params([64, 128, 10]); p_big = count_params([784, 512, 256, 10])
except Exception:
    p_xor = p_dig = p_big = None
check('2-2-1 XOR network has 9 parameters', p_xor == 9)
check('64-128-10 digit network has 9,610 parameters', p_dig == 9610)
check_hash('hidden: parameters of 784-512-256-10', p_big, 'ee542894ac5c53ad')
try:
    f_small = forward_flops([3, 2, 1], 2); f_dig = forward_flops([64, 128, 10], 32)
except Exception:
    f_small = f_dig = None
check('Section 3 example: 32 FLOPs for the two products', f_small == 32)
check_hash('hidden: forward FLOPs of 64-128-10 at batch 32', f_dig, 'caa1917fd26e0599')
try:
    shp = forward_shapes([64, 128, 10], 32)
    shp_ok = [tuple(s) for s in shp] == [(32, 128), (32, 10)]
except Exception:
    shp_ok = False
check('shapes of Z_1, Z_2 for 64-128-10 at batch 32', shp_ok)

## Exercise 3 — Stable log-softmax and cross-entropy  `★☆☆`

Implement `log_softmax(Z)` for an $(n, k)$ logits matrix using the log-sum-exp trick,

$$
\ln P_{ij} = (Z_{ij} - m_i) - \ln \sum_c e^{Z_{ic} - m_i}, \qquad m_i = \max_c Z_{ic},
$$

and `cross_entropy(Z, y)` returning the mean of $-\ln P_{i, y_i}$ over the rows as a `float` (`y` is an integer array).
Both must survive logits such as 1000 and −1000 without `inf` or `nan`. The checks compare against
`scipy.special.log_softmax` and ask for the loss on a hidden logits matrix.

In [ ]:
def log_softmax(Z):
    # row-wise log-softmax of an (n, k) array, computed with the log-sum-exp trick
    # YOUR CODE HERE
    return None

def cross_entropy(Z, y):
    # mean over rows of -log P[i, y_i], as a float; y is an integer array of shape (n,)
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
from scipy.special import log_softmax as _sp_log_softmax
Z3 = np.array([[2.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1000.0, 999.0, -1000.0], [-3.0, 4.0, 1.0], [0.5, 0.5, 5.0]])
y3 = np.array([0, 1, 1, 1, 2])
try:
    lp = log_softmax(Z3); lp = None if lp is None else np.asarray(lp, dtype=float)
except Exception:
    lp = None
check_close('log_softmax matches scipy.special.log_softmax (row-wise)', lp, _sp_log_softmax(Z3, axis=1), tol=1e-9)
check('finite for logits of +-1000', lp is not None and bool(np.all(np.isfinite(lp))))
check_close('rows exponentiate to probabilities summing to 1', (np.exp(lp).sum(axis=1) if lp is not None else None), np.ones(5), tol=1e-9)
try:
    ce = cross_entropy(Z3, y3); ce_uniform = cross_entropy(np.zeros((4, 10)), np.array([0, 3, 5, 9]))
except Exception:
    ce = ce_uniform = None
check_close('uniform logits over 10 classes give loss ln 10', ce_uniform, np.log(10), tol=1e-9)
check('cross_entropy returns a Python float', isinstance(ce, float))
check_hash('hidden: cross-entropy on the test logits', ce, '9ad30fe0a57d8ea7')

## Exercise 4 — The gradient of softmax cross-entropy  `★★☆`

Implement `softmax_ce_grad(Z, y)` returning $\partial \mathcal{L}/\partial Z = (P - Y)/n$ for the mean cross-entropy
of Exercise 3 ($Y$ is the one-hot matrix of `y`), and `sigmoid_mse_grad(z, y)` returning the elementwise derivative
of $(\sigma(z) - y)^2$ with respect to $z$, i.e. $2(\sigma(z) - y)\,\sigma(z)(1 - \sigma(z))$.

The checks verify the first against central differences of your `cross_entropy` and ask, as a hidden answer rounded
to one decimal, for the ratio of the cross-entropy gradient $\sigma(z) - y$ to the squared-error gradient at $z = -5$, $y = 1$.

In [ ]:
def softmax_ce_grad(Z, y):
    # dL/dZ for L = cross_entropy(Z, y): shape (n, k). Use log_softmax from Exercise 3.
    # YOUR CODE HERE
    return None

def sigmoid_mse_grad(z, y):
    # dL/dz for L = (sigmoid(z) - y)**2, elementwise (for the comparison plot / check)
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    G4 = softmax_ce_grad(Z3, y3); G4 = None if G4 is None else np.asarray(G4, dtype=float)
except Exception:
    G4 = None
check('gradient has the shape of Z', G4 is not None and G4.shape == Z3.shape)
check_close('gradient rows sum to zero', (G4.sum(axis=1) if G4 is not None else None), np.zeros(5), tol=1e-9)
def _num_grad(Z, y, h=1e-6):
    g = np.zeros_like(Z)
    for idx in np.ndindex(Z.shape):
        Zp = Z.copy(); Zp[idx] += h; Zm = Z.copy(); Zm[idx] -= h
        g[idx] = (cross_entropy(Zp, y) - cross_entropy(Zm, y)) / (2 * h)
    return g
try:
    Zr = rng.normal(size=(6, 4)) * 2; yr = rng.integers(0, 4, size=6)
    ng = _num_grad(Zr, yr); ag = np.asarray(softmax_ce_grad(Zr, yr), dtype=float)
except Exception:
    ng = ag = None
check_close('(P - Y)/n matches central differences of cross_entropy', ag, ng, tol=1e-6)
check_hash('hidden: sum |gradient| on the test logits', (float(np.abs(G4).sum()) if G4 is not None else None), '3d719c7957a8a263')
try:
    ratio = float((sigmoid(-5.0) - 1.0) / sigmoid_mse_grad(-5.0, 1.0))
    mse0 = float(sigmoid_mse_grad(0.0, 1.0))
except Exception:
    ratio = mse0 = None
check_close('sigmoid-MSE gradient at z = 0, y = 1 is -0.25', mse0, -0.25)
check_hash('hidden: CE/MSE gradient ratio at z = -5 (1 decimal)', ratio, '8db5dccea6502496', decimals=1)

## Exercise 5 — `Dense.backward` and a gradient checker  `★★☆`

Complete `Dense.backward(G)`: given $G = \partial\mathcal{L}/\partial Z$ for $Z = XW + \mathbf 1 b^\top$, store
$\partial\mathcal{L}/\partial W = X^\top G$ in `self.dW` and $\partial\mathcal{L}/\partial b = \sum_{\text{rows}} G$ in `self.db`, and return
$\partial\mathcal{L}/\partial X = G W^\top$. `ReLU` and `MLP` are given.

Then write `grad_check(model, loss_fn, Xb, yb, h=1e-5)`: run `model.forward`, `loss_fn` (which returns `(loss, dL/dZ)`)
and `model.backward` once, flatten the analytic gradients `model.grads()` in order, estimate every parameter entry
by central differences (perturb `p[idx]` in place, evaluate the loss, restore), and return the relative error
$\lVert g - \hat g\rVert / (\lVert g\rVert + \lVert \hat g\rVert)$ as a float. The hidden check is $\partial\mathcal{L}/\partial W_1$ for the concept guide's
Section 5 example.

In [ ]:
class Dense:
    def __init__(self, n_in, n_out, scale=None, rng=rng):
        scale = np.sqrt(2.0 / n_in) if scale is None else scale
        self.W = rng.normal(0.0, scale, size=(n_in, n_out))
        self.b = np.zeros(n_out)
        self.dW = np.zeros_like(self.W); self.db = np.zeros_like(self.b)
    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, G):
        # G = dL/dZ with Z = X @ W + b, shape (n, n_out). Store dL/dW in self.dW and dL/db in self.db,
        # and return dL/dX (shape (n, n_in)).
        # YOUR CODE HERE
        return None

    def params(self): return [self.W, self.b]
    def grads(self):  return [self.dW, self.db]

class ReLU:
    def forward(self, Z):
        self.mask = Z > 0
        return Z * self.mask
    def backward(self, G): return G * self.mask
    def params(self): return []
    def grads(self):  return []

class MLP:
    def __init__(self, layers): self.layers = layers
    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X
    def backward(self, G):
        for layer in reversed(self.layers):
            G = layer.backward(G)
        return G
    def params(self): return [p for l in self.layers for p in l.params()]
    def grads(self):  return [g for l in self.layers for g in l.grads()]

def grad_check(model, loss_fn, Xb, yb, h=1e-5):
    # loss_fn(Z, yb) -> (loss, dL/dZ). Run forward/backward once to get the analytic gradients of all
    # model.params(), estimate every entry by central differences, and return the relative error
    # ||g - g_num|| / (||g|| + ||g_num||) as a float.
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
X5 = np.array([[1., 2., 0.], [0., 1., -1.]]); y5 = np.array([[3.], [1.]])
mse5 = lambda Z, y: (0.5 * np.sum((Z - y) ** 2) / len(Z), (Z - y) / len(Z))
try:
    L1 = Dense(3, 2); L1.W[:] = np.array([[0.5, -1.], [1., 0.5], [0., 2.]]); L1.b[:] = np.array([0.1, 0.2])
    L2 = Dense(2, 1); L2.W[:] = np.array([[1.], [-1.]]); L2.b[:] = np.array([0.5])
    net5 = MLP([L1, ReLU(), L2])
    Z5 = net5.forward(X5); loss5, G5 = mse5(Z5, y5); dX5 = net5.backward(G5)
    dW1, db1, dW2, db2 = L1.dW.copy(), L1.db.copy(), L2.dW.copy(), L2.db.copy()
except Exception:
    dW1 = db1 = dW2 = db2 = dX5 = None
check_close('dW2 = (0.2, -0.01)^T', dW2, np.array([[0.2], [-0.01]]), tol=1e-9)
check_close('db2 = 0.25', db2, np.array([0.25]), tol=1e-9)
check_close('db1 = (0.25, 0.05)', db1, np.array([0.25, 0.05]), tol=1e-9)
check('dL/dX has the shape of X', dX5 is not None and np.shape(dX5) == X5.shape)
check_hash('hidden: dW1 of the Section 5 example', dW1, '2955836d45185e83')
from scipy.special import log_softmax as _lsm
def _softmax_ce(Z, y):                       # reference loss for the checks (uses scipy, not your Exercise 3 code)
    n = Z.shape[0]; logp = _lsm(Z, axis=1)
    G = np.exp(logp); G[np.arange(n), y] -= 1.0
    return -logp[np.arange(n), y].mean(), G / n
try:
    r5 = np.random.default_rng(5)
    net_r = MLP([Dense(7, 6, rng=r5), ReLU(), Dense(6, 5, rng=r5), ReLU(), Dense(5, 3, rng=r5)])
    rel = grad_check(net_r, _softmax_ce, r5.normal(size=(8, 7)), r5.integers(0, 3, size=8))
except Exception:
    rel = None
check('grad_check on a random 7-6-5-3 network: relative error < 1e-7', rel is not None and rel < 1e-7)
class _BuggyDense(Dense):                     # your layer, with its bias gradient deliberately corrupted
    def backward(self, G):
        out = super().backward(G)
        self.db = self.db / G.shape[0]           # mean instead of sum: a classic wrong bias rule
        return out
try:
    r5b = np.random.default_rng(5)
    net_b = MLP([_BuggyDense(7, 6, rng=r5b), ReLU(), _BuggyDense(6, 3, rng=r5b)])
    rel_b = grad_check(net_b, _softmax_ce, r5b.normal(size=(8, 7)), r5b.integers(0, 3, size=8))
except Exception:
    rel_b = None
check('grad_check catches a planted bug (relative error > 1e-2)', rel_b is not None and rel_b > 1e-2)

## Exercise 6 — He initialisation and variance propagation  `★★☆`

Implement `he_std(n_in)` $= \sqrt{2/n_{\text{in}}}$ and `xavier_std(n_in, n_out)` $= \sqrt{2/(n_{\text{in}} + n_{\text{out}})}$, and
`propagate_std(scale, depth=10, width=256, n=1000, act="relu", seed=0)`, which pushes a standard-normal batch
`A = r.normal(size=(n, width))` (with `r = np.random.default_rng(seed)`) through `depth` layers, each drawing
`W = r.normal(0.0, scale, size=(width, width))` *after* `A` and setting `A = act(A @ W)`, and returns the standard
deviation of the final activations as a float.

Use it to confirm the variance-propagation rule: with He scaling the ReLU activations keep a standard deviation of
order 1 after 10 layers, with scale 0.05 ($n_{\text{in}} \sigma_w^2 = 0.64$) they shrink, and with scale 0.15 ($n_{\text{in}} \sigma_w^2 = 5.76$)
they grow. The hidden check asks for `he_std(300)`.

In [ ]:
def he_std(n_in):
    # standard deviation of He initialisation for a layer with n_in inputs
    # YOUR CODE HERE
    return None

def xavier_std(n_in, n_out):
    # standard deviation of Xavier/Glorot initialisation
    # YOUR CODE HERE
    return None

def propagate_std(scale, depth=10, width=256, n=1000, act="relu", seed=0):
    # Start from A = r.normal(size=(n, width)) with r = np.random.default_rng(seed). For each of `depth` layers,
    # draw W = r.normal(0.0, scale, size=(width, width)) and set A = act(A @ W) (act: "relu" or "tanh").
    # Return the standard deviation of the final A as a float. Draw W AFTER A, in that order, so the
    # checks (which rerun this recipe) see the same random numbers.
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    he256 = he_std(256); xa = xavier_std(100, 300); he300 = he_std(300)
except Exception:
    he256 = xa = he300 = None
check_close('he_std(256) = sqrt(2/256)', he256, np.sqrt(2 / 256), tol=1e-9)
check_close('xavier_std(100, 300) = sqrt(2/400)', xa, np.sqrt(2 / 400), tol=1e-9)
check_hash('hidden: he_std(300)', he300, 'f42a4f07654eac2c')
try:
    s_he = propagate_std(np.sqrt(2 / 256)); s_small = propagate_std(0.05); s_big = propagate_std(0.15)
    s_tanh_x = propagate_std(np.sqrt(1 / 256), act="tanh")
except Exception:
    s_he = s_small = s_big = s_tanh_x = None
check('He scale: ReLU activation std after 10 layers stays between 0.5 and 1.5', s_he is not None and 0.5 < s_he < 1.5)
check('scale 0.05 (n*sigma^2 = 0.64): std shrinks below 0.05', s_small is not None and s_small < 0.05)
check('scale 0.15 (n*sigma^2 = 5.76): std grows above 10', s_big is not None and s_big > 10)
check('Xavier scale with tanh: std stays between 0.1 and 1', s_tanh_x is not None and 0.1 < s_tanh_x < 1.0)
check_close('propagate_std follows the specified random recipe', s_he, 0.8833058456361979, tol=1e-6)

## Exercise 7 — Batch normalisation, forward and backward  `★★☆`

Implement `batchnorm_forward(Z, gamma, beta, eps=1e-5)` returning `(Y, cache)` with the per-column formulas

$$
\mu_j = \tfrac1n\sum_i Z_{ij},\quad \sigma_j^2 = \tfrac1n\sum_i (Z_{ij} - \mu_j)^2,\quad
\hat Z_{ij} = \frac{Z_{ij} - \mu_j}{\sqrt{\sigma_j^2 + \epsilon}},\quad Y_{ij} = \gamma_j \hat Z_{ij} + \beta_j,
$$

and `batchnorm_backward(G, cache)` returning `(dZ, dgamma, dbeta)` for an upstream gradient $G = \partial\mathcal{L}/\partial Y$:

$$
\frac{\partial\mathcal{L}}{\partial\gamma_j} = \sum_i G_{ij}\hat Z_{ij}, \qquad
\frac{\partial\mathcal{L}}{\partial\beta_j} = \sum_i G_{ij}, \qquad
\frac{\partial\mathcal{L}}{\partial Z_{ij}} = \frac{\gamma_j}{n\sqrt{\sigma_j^2 + \epsilon}}\Big(nG_{ij} - \sum_{i'} G_{i'j} - \hat Z_{ij}\sum_{i'} G_{i'j}\hat Z_{i'j}\Big).
$$

The checks reproduce the concept guide's worked example, verify the scale invariance, and compare `dZ` with central
differences of $\sum G \odot Y$ — the backward pass is *not* elementwise, because $\mu$ and $\sigma^2$ depend on every row.

In [ ]:
def batchnorm_forward(Z, gamma, beta, eps=1e-5):
    # Z: (n, h). Per column: mu, biased variance, Zhat = (Z - mu) / sqrt(var + eps), Y = gamma * Zhat + beta.
    # Return (Y, cache) where cache holds whatever the backward pass needs (e.g. Zhat, 1/sqrt(var + eps), gamma).
    # YOUR CODE HERE
    return None, None

def batchnorm_backward(G, cache):
    # G = dL/dY, shape (n, h). Return (dZ, dgamma, dbeta) with
    #   dgamma = sum_rows(G * Zhat),  dbeta = sum_rows(G),
    #   dZ = gamma / (n * sqrt(var + eps)) * (n * G - sum_rows(G) - Zhat * sum_rows(G * Zhat)).
    # YOUR CODE HERE
    return None, None, None

**Check your answer:**

In [ ]:
Z7 = np.array([[1.], [2.], [3.], [6.]])
try:
    Y7, c7 = batchnorm_forward(Z7, np.array([2.0]), np.array([1.0]))
    Y7 = None if Y7 is None else np.asarray(Y7, dtype=float)
    Y7_scaled = np.asarray(batchnorm_forward(100 * Z7, np.array([2.0]), np.array([1.0]))[0], dtype=float)
except Exception:
    Y7 = Y7_scaled = None
check_close('worked example: batch (1, 2, 3, 6), gamma 2, beta 1', (Y7.ravel() if Y7 is not None else None), 2 * (Z7.ravel() - 3) / np.sqrt(3.5 + 1e-5) + 1, tol=1e-9)
check_close('scale invariance: inputs x100 give (almost) the same output', Y7_scaled, Y7, tol=1e-4)
r7 = np.random.default_rng(7)
Zb = r7.normal(size=(6, 4)) * 3 + 1; gm = r7.normal(size=4); bt = r7.normal(size=4); Gb = r7.normal(size=(6, 4))
try:
    Yb, cb = batchnorm_forward(Zb, gm, bt)
    dZ, dg, db = batchnorm_backward(Gb, cb)
    dZ = np.asarray(dZ, dtype=float); dg = np.asarray(dg, dtype=float); db = np.asarray(db, dtype=float)
    Zhat_ref = (Zb - Zb.mean(0)) / np.sqrt(Zb.var(0) + 1e-5)
except Exception:
    dZ = dg = db = None; Zhat_ref = None
try:
    col_stats = np.r_[np.asarray(Yb, dtype=float).mean(0), np.asarray(Yb, dtype=float).std(0)]
except Exception:
    col_stats = None
check_close('output columns have mean beta and std |gamma|', col_stats, np.r_[bt, np.abs(gm) * np.sqrt(Zb.var(0) / (Zb.var(0) + 1e-5))], tol=1e-6)
check_close('dgamma = sum(G * Zhat)', dg, ((Gb * Zhat_ref).sum(0) if Zhat_ref is not None else None), tol=1e-9)
check_close('dbeta = sum(G)', db, Gb.sum(0), tol=1e-9)
def _num_dZ(h=1e-6):
    g = np.zeros_like(Zb)
    for idx in np.ndindex(Zb.shape):
        Zp = Zb.copy(); Zp[idx] += h; Zm = Zb.copy(); Zm[idx] -= h
        g[idx] = (np.sum(Gb * batchnorm_forward(Zp, gm, bt)[0]) - np.sum(Gb * batchnorm_forward(Zm, gm, bt)[0])) / (2 * h)
    return g
try:
    ndZ = _num_dZ() if dZ is not None else None
except Exception:
    ndZ = None
check_close('dZ matches central differences', dZ, ndZ, tol=1e-6)
check_close('dZ columns sum to zero (the mean is removed)', (dZ.sum(0) if dZ is not None else None), np.zeros(4), tol=1e-9)
check_hash('hidden: sum |dZ| on the random batch', (float(np.abs(dZ).sum()) if dZ is not None else None), 'eefc8436ea77b42a')

## Exercise 8 — Inverted dropout keeps the expectation  `★★☆`

Implement `dropout_forward(A, p, rng, training=True)` (inverted dropout: in training mode keep each entry with
probability $1 - p$ via `mask = (rng.random(A.shape) >= p) / (1 - p)` and return `(A * mask, mask)`; in evaluation
mode, or when `p == 0`, return `A` unchanged with a mask of ones) and `dropout_backward(G, mask)` $= G \odot \text{mask}$.

The checks verify that the surviving entries are scaled by $1/(1 - p)$, that about a fraction $p$ of the entries are
zero, that the *mean over many masks* of the dropped activations equals the original (within sampling error), and
that evaluation mode is the identity.

In [ ]:
def dropout_forward(A, p, rng, training=True):
    # Inverted dropout. In training mode draw mask = (rng.random(A.shape) >= p) / (1 - p) and return
    # (A * mask, mask); in evaluation mode (or p == 0) return (A, ones_like(A)).
    # YOUR CODE HERE
    return None, None

def dropout_backward(G, mask):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
A8 = np.array([2.0, -1.0, 3.0, 0.5])
try:
    out, mask = dropout_forward(A8, 0.5, np.random.default_rng(0))
    out = np.asarray(out, dtype=float); mask = np.asarray(mask, dtype=float)
    kept = mask > 0
    scale_ok = np.allclose(out[kept], 2.0 * A8[kept]) and np.allclose(out[~kept], 0.0)
    out_eval, mask_eval = dropout_forward(A8, 0.5, np.random.default_rng(0), training=False)
    eval_ok = np.allclose(out_eval, A8) and np.allclose(mask_eval, 1.0)
except Exception:
    scale_ok = eval_ok = False; mask = None
check('survivors are scaled by 1/(1-p), the rest are zero', bool(scale_ok))
check('evaluation mode is the identity', bool(eval_ok))
try:
    r8 = np.random.default_rng(8)
    big = np.ones((200, 500))
    outs = [dropout_forward(big, 0.3, r8)[0] for _ in range(1)][0]
    frac_zero = float(np.mean(outs == 0))
    mean_val = float(outs.mean())
    A_rand = r8.normal(size=(50, 40))
    avg = np.mean([dropout_forward(A_rand, 0.5, r8)[0] for _ in range(400)], axis=0)
    max_dev = float(np.max(np.abs(avg - A_rand)))
except Exception:
    frac_zero = mean_val = max_dev = None
check('about 30% of entries are dropped at p = 0.3', frac_zero is not None and 0.28 < frac_zero < 0.32)
check('mean of the dropped activations stays ~1 (expectation preserved)', mean_val is not None and 0.98 < mean_val < 1.02)
check('average over 400 masks reproduces A to within 0.4', max_dev is not None and max_dev < 0.4)
try:
    G8 = np.array([1.0, 1.0, 1.0, 1.0]); back = np.asarray(dropout_backward(G8, mask), dtype=float)
    back_ok = np.allclose(back, mask)
except Exception:
    back_ok = False
check('backward multiplies by the same mask', bool(back_ok))

## Exercise 9 — Depth without nonlinearity, and the factor 2  `★★☆`

1. A colleague proposes a 20-layer network with *no* activation functions, arguing that "each layer learns a
   progressively more abstract linear feature". Show in two lines what function class the network can represent and
   what the extra 19 layers buy.
2. Explain where the factor 2 in He initialisation, $\sigma_w^2 = 2/n_{\text{in}}$, comes from, starting from
   $\operatorname{Var}(z) = n_{\text{in}} \sigma_w^2 \sigma_x^2$. What would the factor be for a leaky ReLU with negative slope $\alpha = 0.2$?
3. Your 12-layer tanh network trains fine with Xavier initialisation. You replace tanh by ReLU and keep Xavier. The
   loss decreases much more slowly. Predict the activation standard deviation at layer 12 relative to layer 1 and explain.

<details><summary><b>Show answer</b></summary>

1. $(\cdots((XW_1 + b_1)W_2 + b_2)\cdots)W_{20} + b_{20} = X(W_1 W_2 \cdots W_{20}) + b'$: a single affine map. The network
   can represent exactly what linear (or logistic) regression can — one hyperplane — with 19 redundant factorisations
   of one matrix, worse conditioning, and more parameters. Depth adds expressive power only through a non-polynomial
   nonlinearity between the layers.
2. A ReLU zeroes the negative half of a symmetric pre-activation, so $E[\mathrm{ReLU}(z)^2] = \tfrac12 E[z^2]$. The next layer's
   pre-activation variance is therefore $n_{\text{in}} \sigma_w^2 \cdot \tfrac12 \sigma_z^2$, and keeping it equal to $\sigma_z^2$ requires
   $n_{\text{in}}\sigma_w^2 = 2$. For a leaky ReLU the negative half contributes $\alpha^2 \cdot \tfrac12 E[z^2]$ instead of zero, so the
   second moment is $\tfrac{1 + \alpha^2}{2} E[z^2]$ and the factor becomes $2/(1 + \alpha^2) = 2/1.04 \approx 1.92$ for $\alpha = 0.2$.
3. Xavier gives $n_{\text{in}}\sigma_w^2 \approx 1$, which preserves variance through a (near-linear) tanh but not through a ReLU,
   whose halving makes the per-layer variance factor $\tfrac12$. After 11 more layers the variance is $2^{-11}$ of its
   initial value, so the standard deviation is about $2^{-5.5} \approx 0.02$ of layer 1's. The gradients flow back through
   the same matrices and shrink by the same factor, so the early layers barely move. Switching to He scaling
   ($\sigma_w^2 = 2/n_{\text{in}}$) restores the factor to 1.

</details>

## Exercise 10 — Convolution as a Toeplitz matrix  `★★★`

Implement `conv_matrix(w, n)`, the $(n - m + 1) \times n$ Toeplitz matrix $T$ with `T[i, i:i+m] = w` for a kernel $w$ of
length $m$, so that `T @ x` is the "valid" cross-correlation $y_i = \sum_{j=0}^{m-1} w_j x_{i+j}$; `conv1d(x, w)` using it; and
`conv1d_backward(x, w, G)` returning `(dx, dw)` for an upstream gradient $G = \partial\mathcal{L}/\partial y$:

$$
\frac{\partial\mathcal{L}}{\partial x} = T^\top G, \qquad \frac{\partial\mathcal{L}}{\partial w_j} = \sum_i G_i\, x_{i+j}
$$

(the shared weight collects one contribution per output position). The checks compare `conv1d` with
`np.convolve(x, w[::-1], mode="valid")` (NumPy flips the kernel, so we flip it back), compare `dx` and `dw` with
central differences, and ask for a hidden `dw`.

In [ ]:
def conv_matrix(w, n):
    # the (n - m + 1, n) Toeplitz matrix T with T[i, i:i+m] = w, so that T @ x is the valid cross-correlation of x with w
    # YOUR CODE HERE
    return None

def conv1d(x, w):
    # y_i = sum_j w_j x_{i+j}, as a NumPy array of length n - m + 1 (use conv_matrix)
    # YOUR CODE HERE
    return None

def conv1d_backward(x, w, G):
    # G = dL/dy. Return (dL/dx, dL/dw): dx = T^T G, dw_j = sum_i G_i x_{i+j}
    # YOUR CODE HERE
    return None, None

**Check your answer:**

In [ ]:
x10 = np.array([1., 2., 3., 4., 2., 0., -1.]); w10 = np.array([1., -1., 0.5])
try:
    T = conv_matrix(w10, len(x10)); T = None if T is None else np.asarray(T, dtype=float)
    y10 = conv1d(x10, w10); y10 = None if y10 is None else np.asarray(y10, dtype=float)
except Exception:
    T = y10 = None
check('T has shape (n - m + 1, n) = (5, 7)', T is not None and T.shape == (5, 7))
try:
    structure_ok = T.shape == (5, 7) and np.all((T != 0).sum(axis=1) == 3) and np.allclose(T[0, :3], w10) and np.allclose(T[4, 4:], w10)
except Exception:
    structure_ok = False
check('T has exactly 3 non-zeros per row, shared along diagonals', bool(structure_ok))
check_close('conv1d matches np.convolve with the flipped kernel', y10, np.convolve(x10, w10[::-1], mode="valid"), tol=1e-12)
G10 = np.array([0.5, -1.0, 2.0, 0.0, 1.5])
try:
    dx, dw = conv1d_backward(x10, w10, G10)
    dx = None if dx is None else np.asarray(dx, dtype=float); dw = None if dw is None else np.asarray(dw, dtype=float)
except Exception:
    dx = dw = None
def _num(f, v, h=1e-6):
    g = np.zeros_like(v)
    for i in range(len(v)):
        vp = v.copy(); vp[i] += h; vm = v.copy(); vm[i] -= h
        g[i] = (f(vp) - f(vm)) / (2 * h)
    return g
ref_loss_x = lambda xx: float(G10 @ np.convolve(xx, w10[::-1], mode="valid"))
ref_loss_w = lambda ww: float(G10 @ np.convolve(x10, ww[::-1], mode="valid"))
check_close('dL/dx = T^T G matches central differences', dx, _num(ref_loss_x, x10), tol=1e-6)
check_close('dL/dx equals the full convolution of G with the kernel', dx, np.convolve(G10, w10, mode="full"), tol=1e-9)
check_close('dL/dw matches central differences', dw, _num(ref_loss_w, w10), tol=1e-6)
try:
    _, dw_ones = conv1d_backward(x10, w10, np.ones(5)); dw_ones = None if dw_ones is None else np.asarray(dw_ones, dtype=float)
except Exception:
    dw_ones = None
check_hash('hidden: dL/dw for G = ones', dw_ones, '1f90d0c4f7b3ca05')

## Exercise 11 — Mini-project: a regularised digit classifier with a confidence interval  `★★★`

Train your own NumPy MLP on the handwritten digits and report the result the way a careful practitioner would.

```python
from sklearn.datasets import load_digits
digits = load_digits(); X_all, y_all = digits.data, digits.target
```

1. Standardise the pixels with training-set statistics (set the standard deviation of constant pixels to 1) and
   split 80/20 with a fixed seed.
2. Build a 64 → 128 → 10 network from the `Dense`, `ReLU` and `MLP` classes of Exercise 5 (He initialisation for the
   hidden layer, a small scale such as 0.01 for the output layer so the initial loss is close to $\ln 10$), using
   `log_softmax` from Exercise 3 for the loss and `softmax_ce_grad` from Exercise 4 for its gradient, and a `Dropout`
   layer built on Exercise 8's functions. Gradient-check the untrained model on a tiny batch with your `grad_check`.
3. Write an Adam (or SGD with momentum) optimiser with optional decoupled weight decay and a mini-batch training
   loop (shuffle every epoch; switch dropout off for evaluation). Train four variants for about 30 epochs: plain,
   dropout 0.3, weight decay $10^{-2}$, both.
4. For each variant report the test accuracy with a 95 percent **bootstrap confidence interval** (Module 10: resample
   the per-example correct/incorrect indicators 2,000 times) and plot the training and test loss curves.
5. Visualise 16 columns of the first-layer weight matrix as $8 \times 8$ images — they are pixel templates.
6. State in one sentence whether the differences between variants are resolvable on 360 test images.

**Rubric — you have mastered this when you can:**

- The model is built only from the NumPy classes and functions of this notebook (no frameworks), and the gradient check on the untrained model reports a relative error below $10^{-6}$.
- Standardisation uses training-set statistics only, and the initial loss is within 0.1 of $\ln 10 \approx 2.30$.
- The training loop reshuffles every epoch, applies dropout only in training mode, and reaches at least 95 percent test accuracy for every variant.
- Each accuracy is reported with a bootstrap confidence interval, and the conclusion about the variants refers to the overlap of those intervals rather than to the point estimates.
- The loss curves have titles and axis labels, use a log scale for the loss, and show the regularised variants with a higher training loss than the plain model.
- The first-layer weight images are plotted with a symmetric colour scale, and you can say in one sentence what a column of $W_1$ means.

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits(); X_all, y_all = digits.data, digits.target

# Your mini-project workspace. You can reuse Dense / ReLU / MLP / grad_check (Exercise 5),
# log_softmax (Exercise 3), softmax_ce_grad (Exercise 4) and dropout_forward/backward (Exercise 8),
# which are all defined above in this notebook.

## Score

In [ ]:
summary()